### Global-level

In [4]:
import json
import itertools
from pathlib import Path
import pandas as pd
import os
import numpy as np
from scipy.stats import pearsonr  # Essential for the new correlation logic

# --- Configuration ---
OUTPUT_ROOT = "/Users/javigg/Desktop/h2t_correlations/evaluation/output_evals"

DATASETS = [
    "fleurs", "covost2", "europarl_st", "cs_fleurs", "mexpresso", 
    "acl6060-short", "acl6060-long", "mcif-long", "mcif-short"
]

DIRECTION_PAIRS = [
    "en_de","de_en","en_es","es_en","en_fr","fr_en","en_it","it_en",
    "en_nl","nl_en","en_pt","pt_en","en_zh","zh_en"
]

SYSTEM_NAMES = ['whisper', 'seamlessm4t', 'canary-v2', 'owsm4.0-ctc',
                'aya_whisper', 'gemma_whisper', 'tower_whisper', 
                'aya_seamlessm4t', 'gemma_seamlessm4t', 'tower_seamlessm4t',
                'aya_canary-v2', 'gemma_canary-v2', 'tower_canary-v2',
                'aya_owsm4.0-ctc', 'gemma_owsm4.0-ctc', 'tower_owsm4.0-ctc',
                'desta2-8b', 'qwen2audio-7b', 'phi4multimodal', 'voxtral-small-24b', 'spirelm'
               ]

SELECTED_COLS = [
    "dataset", "direction", "system",
    "SacreBLEU", "chrF", "LinguaPy",
    "RefMetricX_24", "RefMetricX_24-Strict-linguapy", "QEMetricX_24-Strict-linguapy",
    "XCOMET-Strict-linguapy", "XCOMET", "XCOMET-QE-Strict-linguapy", "QEMetricX_24", "XCOMET-QE"
]

METRICX_REF = "RefMetricX_24-Strict-linguapy"
METRICX_QE  = "QEMetricX_24-Strict-linguapy"
XCOMET_REF  = "XCOMET-Strict-linguapy"
XCOMET_QE   = "XCOMET-QE-Strict-linguapy"

# --- Data Loading Functions ---

def load_results_summaries(base_dir, direction_pairs, system_names):
    base_path = Path(base_dir)
    all_results = {}

    for direction, system in itertools.product(direction_pairs, system_names):
        summary_path = base_path / system / direction / 'results_summary.jsonl'
        
        if direction not in all_results:
            all_results[direction] = {}

        try:
            with summary_path.open('r', encoding='utf-8') as f:
                all_results[direction][system] = [json.loads(line) for line in f]
        except FileNotFoundError:
            all_results[direction][system] = None
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON in {summary_path}: {e}")
            all_results[direction][system] = None

    return all_results

def convert_results_to_dataframe(results_data):
    all_records = [
        {
            'direction': direction,
            'system': system,
            **record 
        }
        for direction, systems in results_data.items()
        for system, records in systems.items()
        if records is not None
        for record in records
    ]

    if not all_records:
        return pd.DataFrame()

    df = pd.DataFrame(all_records)
    original_cols = [col for col in df.columns if col not in ['direction', 'system']]
    preferred_order = ['direction', 'system'] + original_cols
    return df[preferred_order]

def load_dataset_df(dataset_name: str) -> pd.DataFrame:
    base_dir = os.path.join(OUTPUT_ROOT, dataset_name)
    results_data = load_results_summaries(base_dir, DIRECTION_PAIRS, SYSTEM_NAMES)
    results_df = convert_results_to_dataframe(results_data)
    results_df = results_df.copy()
    results_df["dataset"] = dataset_name
    return results_df

# --- Correlation Logic ---

def compute_qe_ref_correlations(df: pd.DataFrame,
                                dataset_col: str = "dataset",
                                direction_col: str = "direction") -> pd.DataFrame:
    """
    Computes Pearson correlation and p-values using scipy.stats.pearsonr.
    """
    results = []
    group_cols = [dataset_col, direction_col] if dataset_col in df.columns else [direction_col]

    for keys, sub in df.groupby(group_cols):
        if isinstance(keys, tuple):
            dataset, direction = keys
        else:
            dataset, direction = (None, keys)

        def get_stats(data, col_ref, col_qe):
            # Drop rows where either metric is missing
            valid_sub = data[[col_ref, col_qe]].dropna()
            # Pearson requires at least 2 data points for a correlation
            if len(valid_sub) > 1:
                # scipy returns (correlation_coefficient, p_value)
                r_val, p_val = pearsonr(valid_sub[col_ref], valid_sub[col_qe])
                return r_val, p_val, len(valid_sub)
            return np.nan, np.nan, len(valid_sub)

        mx_r, mx_p, mx_n = get_stats(sub, METRICX_REF, METRICX_QE)
        xc_r, xc_p, xc_n = get_stats(sub, XCOMET_REF, XCOMET_QE)

        results.append({
            "dataset": dataset if dataset_col in df.columns else "ALL",
            "language_pair": direction,
            "mx_pearson": mx_r,
            "mx_p_val": mx_p,
            "xc_pearson": xc_r,
            "xc_p_val": xc_p,
            "n_mx": mx_n,
            "n_xc": xc_n,
        })

    return pd.DataFrame(results).sort_values(["dataset", "language_pair"])

# --- Main Execution ---

all_dfs = []
for ds in DATASETS:
    try:
        all_dfs.append(load_dataset_df(ds))
    except Exception as e:
        print(f"[WARN] Failed loading dataset={ds}: {e}")

if not all_dfs:
    raise RuntimeError("No datasets were loaded.")

results_df_all = pd.concat(all_dfs, ignore_index=True)
existing_cols = [c for c in SELECTED_COLS if c in results_df_all.columns]
df_filtered = results_df_all[existing_cols].copy()

# Compute correlations with scipy
corr_table = compute_qe_ref_correlations(df_filtered)

print("### Detailed Pearson Correlations (per Dataset & Direction) ###")
with pd.option_context("display.max_rows", None, "display.precision", 4):
    print(corr_table.to_string(index=False))

# Dataset-level summary with Significance Count
summary = (
    corr_table
    .groupby("dataset", as_index=False)
    .agg(
        avg_mx_pearson=("mx_pearson", "mean"),
        mx_sig_count=("mx_p_val", lambda x: (x < 0.05).sum()),
        avg_xc_pearson=("xc_pearson", "mean"),
        xc_sig_count=("xc_p_val", lambda x: (x < 0.05).sum()),
        total_directions=("language_pair", "count")
    )
    .sort_values("dataset")
)

print("\n=== Dataset-level summary (Mean Correlation & Significance) ===")
print(summary.to_string(index=False))

### Detailed Pearson Correlations (per Dataset & Direction) ###
      dataset language_pair  mx_pearson   mx_p_val  xc_pearson   xc_p_val  n_mx  n_xc
 acl6060-long         en_de      0.9996 1.3139e-21      0.9996 9.0882e-22    15    15
 acl6060-long         en_fr      0.9997 1.5636e-22      0.9996 1.3106e-21    15    15
 acl6060-long         en_pt      0.9999 3.8433e-26      0.9998 3.4121e-24    15    15
 acl6060-long         en_zh      0.9999 4.4415e-25      1.0000 1.6066e-27    15    15
acl6060-short         en_de      0.9989 3.0652e-25      0.9986 1.4902e-24    20    20
acl6060-short         en_fr      0.9993 5.2224e-27      0.9987 1.2034e-24    20    20
acl6060-short         en_pt      0.9991 5.6772e-26      0.9980 5.6801e-23    20    20
acl6060-short         en_zh      0.9999 2.6894e-35      0.9998 9.9139e-33    20    20
      covost2         de_en      0.9989 2.7986e-25      0.9970 2.1000e-21    20    20
      covost2         en_de      0.9995 7.5081e-29      0.9991 2.6887e-26   

### Item level

In [5]:
import json
import itertools
from pathlib import Path
import pandas as pd
import os
import numpy as np
from scipy.stats import pearsonr

# --- Configuration ---
OUTPUT_ROOT = "/Users/javigg/Desktop/h2t_correlations/evaluation/output_evals"
MANIFEST_ROOT = "/Users/javigg/Desktop/h2t_correlations/manifests" 

DATASETS = [
    "fleurs", "covost2", "europarl_st", "cs_fleurs", "mexpresso", 
    "acl6060-short", "acl6060-long", "mcif-long", "mcif-short"
]

DIRECTION_PAIRS = [
    "en_de","de_en","en_es","es_en","en_fr","fr_en","en_it","it_en",
    "en_nl","nl_en","en_pt","pt_en","en_zh","zh_en"
]

SYSTEM_NAMES = ['whisper', 'seamlessm4t', 'canary-v2', 'owsm4.0-ctc',
                'aya_whisper', 'gemma_whisper', 'tower_whisper', 
                'aya_seamlessm4t', 'gemma_seamlessm4t', 'tower_seamlessm4t',
                'aya_canary-v2', 'gemma_canary-v2', 'tower_canary-v2',
                'aya_owsm4.0-ctc', 'gemma_owsm4.0-ctc', 'tower_owsm4.0-ctc',
                'desta2-8b', 'qwen2audio-7b', 'phi4multimodal', 'voxtral-small-24b', 'spirelm'
               ]

# Map the keys created in 'load_all_jsons' to the correlation logic
METRICX_REF = "metricx_score"
METRICX_QE  = "metricx_qe_score"
XCOMET_REF  = "xcomet_score"
XCOMET_QE   = "xcomet_qe_score"

# --- Data Loading Functions ---

def load_all_jsons(base_dir, manifests_dir, direction_pairs, system_names):
    """
    Loads item-level scores for all systems and directions in a specific dataset.
    """
    base_path = Path(base_dir)
    manifests_path = Path(manifests_dir)
    all_results = {}
    
    for direction, system in itertools.product(direction_pairs, system_names):
        results_path = base_path / system / direction / 'results.jsonl'
        # Convert en_de -> en-de.jsonl for manifest lookup
        direction_aux = '{direction}.jsonl'.format(direction=direction.replace('_', '-'))
        manifest_path = manifests_path / direction_aux

        if direction not in all_results:
            all_results[direction] = {}

        try:
            # Load Results
            if not results_path.exists():
                continue
                
            with results_path.open('r', encoding='utf-8') as f:
                system_results = [json.loads(line) for line in f]
            
            # Load Manifests and Merge
            # Note: This assumes results.jsonl and manifest.jsonl are line-aligned
            if manifest_path.exists():
                with manifest_path.open('r', encoding='utf-8') as f:
                    manifests = [json.loads(line) for line in f]
                
                # Merge logic
                for it, it_manifests in zip(system_results, manifests):
                    # Basic metadata
                    if 'benchmark_metadata' in it_manifests:
                        it_manifests['gender'] = it_manifests['benchmark_metadata'].get('gender', 'N/A')
                    
                    # Extract raw scores
                    # Note: Using safety checks for list access
                    linguapy_val = it['metrics']['linguapy_score']
                    it['linguapy_score'] = linguapy_val[0] if isinstance(linguapy_val, list) else linguapy_val
                    
                    # Apply user-defined filtering logic (0 if linguapy != 0)
                    # Assuming linguapy_score == 0 is the "Valid/Good" state based on prompt logic
                    is_valid_lang = (it['linguapy_score'] == 0)
                    
                    it['xcomet_qe_score'] = it['metrics'].get('xcomet_qe_score', 0) if is_valid_lang else 0
                    it['xcomet_score']    = it['metrics'].get('xcomet_score', 0)    if is_valid_lang else 0
                    it['metricx_qe_score']= it['metrics'].get('metricx_qe_score', 0) if is_valid_lang else 0
                    it['metricx_score']   = it['metrics'].get('metricx_score', 0)    if is_valid_lang else 0
                    
                    it.update(it_manifests)
                    
                all_results[direction][system] = system_results
            else:
                # If manifest missing, skip or keep partial data? Assuming skip based on zip logic above
                pass

        except Exception as e:
            print(f"Error processing {system}/{direction}: {e}")
            continue

    # Flatten to list
    results = []
    for direction in all_results:
        for system in all_results[direction]:
            for item in all_results[direction][system]:
                item['direction'] = direction
                item['system'] = system
                # Safely get sample_id
                meta = item.get('benchmark_metadata', {})
                item['sample_id'] = meta.get('sample_id', 'unknown')
                results.append(item)

    return pd.DataFrame(results)

# --- Correlation Logic ---

def compute_item_level_correlations(df: pd.DataFrame,
                                    dataset_col: str = "dataset",
                                    direction_col: str = "direction") -> pd.DataFrame:
    """
    Computes Pearson correlation at the ITEM level.
    Groups by Dataset + Direction, pools all items from ALL systems, 
    and correlates Ref metric vs QE metric.
    """
    results = []
    group_cols = [dataset_col, direction_col] if dataset_col in df.columns else [direction_col]

    # Group by Dataset and Language Pair
    for keys, sub_df in df.groupby(group_cols):
        if isinstance(keys, tuple):
            dataset, direction = keys
        else:
            dataset, direction = (None, keys)

        def get_stats(data, col_ref, col_qe):
            # 1. Select columns
            if col_ref not in data.columns or col_qe not in data.columns:
                return np.nan, np.nan, 0
                
            # 2. Drop NaNs
            valid_sub = data[[col_ref, col_qe]].dropna()
            
            # 3. Compute Pearson
            if len(valid_sub) > 1:
                r_val, p_val = pearsonr(valid_sub[col_ref], valid_sub[col_qe])
                return r_val, p_val, len(valid_sub)
            return np.nan, np.nan, len(valid_sub)

        # Compute for MetricX
        mx_r, mx_p, mx_n = get_stats(sub_df, METRICX_REF, METRICX_QE)
        # Compute for XCOMET
        xc_r, xc_p, xc_n = get_stats(sub_df, XCOMET_REF, XCOMET_QE)

        results.append({
            "dataset": dataset,
            "language_pair": direction,
            "mx_pearson": mx_r,
            "mx_p_val": mx_p,
            "xc_pearson": xc_r,
            "xc_p_val": xc_p,
            "n_items": mx_n # Number of items used (approx, assuming overlap)
        })

    if not results:
        return pd.DataFrame()
        
    return pd.DataFrame(results).sort_values(["dataset", "language_pair"])

# --- Main Execution ---

all_dfs = []

print(f"Starting data load from {OUTPUT_ROOT}...")

for ds in DATASETS:
    ds_path = os.path.join(OUTPUT_ROOT, ds)
    ds_manifest_path = os.path.join(MANIFEST_ROOT, ds)
    final_manifest_path = ds_manifest_path if os.path.exists(ds_manifest_path) else MANIFEST_ROOT

    try:
        print(f"Loading {ds}...")
        df_ds = load_all_jsons(ds_path, final_manifest_path, DIRECTION_PAIRS, SYSTEM_NAMES)
        
        if not df_ds.empty:
            df_ds["dataset"] = ds
            all_dfs.append(df_ds)
        else:
            print(f" [WARN] No data found for dataset: {ds}")
            
    except Exception as e:
        print(f" [ERROR] Failed loading dataset={ds}: {e}")

if not all_dfs:
    raise RuntimeError("No datasets were loaded. Check paths and JSON structure.")

print("Concatenating all data...")
results_df_all = pd.concat(all_dfs, ignore_index=True)

print(f"Computing Item-Level Correlations on {len(results_df_all)} total rows...")
corr_table = compute_item_level_correlations(results_df_all)

print("\n### Detailed Pearson Correlations (Item-Level per Dataset & Direction) ###")
with pd.option_context("display.max_rows", None, "display.precision", 4):
    print(corr_table.to_string(index=False))

# Dataset-level summary
summary = (
    corr_table
    .groupby("dataset", as_index=False)
    .agg(
        avg_mx_pearson=("mx_pearson", "mean"),
        mx_sig_count=("mx_p_val", lambda x: (x < 0.05).sum()),
        avg_xc_pearson=("xc_pearson", "mean"),
        xc_sig_count=("xc_p_val", lambda x: (x < 0.05).sum()),
        total_directions=("language_pair", "count"),
        total_items_pooled=("n_items", "sum")
    )
    .sort_values("dataset")
)

print("\n=== Dataset-level summary (Mean Item-Level Correlation) ===")
print(summary.to_string(index=False))

Starting data load from /Users/javigg/Desktop/h2t_correlations/evaluation/output_evals...
Loading fleurs...
Loading covost2...
Loading europarl_st...
Loading cs_fleurs...
Loading mexpresso...
Loading acl6060-short...
Loading acl6060-long...
Loading mcif-long...
Loading mcif-short...
Concatenating all data...
Computing Item-Level Correlations on 4611211 total rows...


/var/folders/fh/zcjnpdj11fb36h2njhyc5str0000gn/T/ipykernel_19526/1859899473.py:145: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r_val, p_val = pearsonr(valid_sub[col_ref], valid_sub[col_qe])
/var/folders/fh/zcjnpdj11fb36h2njhyc5str0000gn/T/ipykernel_19526/1859899473.py:145: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r_val, p_val = pearsonr(valid_sub[col_ref], valid_sub[col_qe])
/var/folders/fh/zcjnpdj11fb36h2njhyc5str0000gn/T/ipykernel_19526/1859899473.py:145: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r_val, p_val = pearsonr(valid_sub[col_ref], valid_sub[col_qe])
/var/folders/fh/zcjnpdj11fb36h2njhyc5str0000gn/T/ipykernel_19526/1859899473.py:145: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r_val, p_val = pearsonr(valid_sub[col_ref], valid_sub[col_qe])
/var/folders/fh/zcjnpdj11fb3


### Detailed Pearson Correlations (Item-Level per Dataset & Direction) ###
      dataset language_pair  mx_pearson    mx_p_val  xc_pearson    xc_p_val  n_items
 acl6060-long         en_de      0.9823  8.2013e-55      0.9901  5.3199e-64       75
 acl6060-long         en_fr      0.9668  5.3488e-45      0.9807  1.5707e-53       75
 acl6060-long         en_pt      0.9817  2.2668e-54      0.9804  2.9665e-53       75
 acl6060-long         en_zh      0.9566  7.8180e-41      0.9876  2.1599e-60       75
acl6060-short         en_de      0.9621  0.0000e+00      0.9761  0.0000e+00     8320
acl6060-short         en_fr      0.9716  0.0000e+00      0.9691  0.0000e+00     8320
acl6060-short         en_pt      0.9729  0.0000e+00      0.9728  0.0000e+00     8320
acl6060-short         en_zh      0.9586  0.0000e+00      0.9745  0.0000e+00     8320
      covost2         de_en      0.9508  0.0000e+00      0.9483  0.0000e+00   270220
      covost2         en_de      0.9633  0.0000e+00      0.9665  0.0000e+0